In [2]:
import pandas as pd
df = pd.read_csv("alerts.csv")
df.head()          # 前5行
df.shape           # 期望 (570, 6)
df.dtypes          # 每列类型

date           str
time           str
level          str
service        str
message        str
rt_ms      float64
dtype: object

In [3]:
print(df.shape)

(570, 6)


In [4]:
df["level"].value_counts()

level
INFO     463
WARN      69
ERROR     38
Name: count, dtype: int64

In [5]:
df[df["level"]=="ERROR"]["service"].value_counts()

service
inventory      10
gateway         8
user-center     7
payment         7
order-api       6
Name: count, dtype: int64

In [6]:
err = df[df["level"]=="ERROR"]
err.assign(hour=err["time"].str[:2])["hour"].value_counts().head(3)

hour
17    7
15    6
18    6
Name: count, dtype: int64

In [7]:
len(df[(df["level"]=="WARN") & (df["service"]=="inventory")])

8

In [8]:
rt = df["rt_ms"].dropna().astype(float)
rt.mean(), rt.max()      # 平均/最大响应毫秒
slow = df.dropna(subset=["rt_ms"]).assign(rt=lambda d: d.rt_ms.astype(float)).nlargest(5,"rt")[["time","service","rt"]]
slow                     # 最慢5条请求

,time,service,rt
44,09:14:28,gateway,2300.0
51,09:21:05,inventory,2300.0
92,10:02:25,inventory,2300.0
142,10:52:55,payment,2300.0
158,11:08:31,payment,2300.0


In [9]:
(df[df["level"]=="ERROR"].groupby("service").size()
   .sort_values(ascending=False)
   .to_csv("error_by_service_pandas.csv", header=["count"]))
!cat error_by_service_pandas.csv    # 与 shell 版 error_by_service.txt 对照，数字应一致

service,count
inventory,10
gateway,8
payment,7
user-center,7
order-api,6
